In [1]:
import os

# 获取当前工作目录
current_dir = os.getcwd()
print("当前工作目录：", current_dir)

# 切换到上一层目录
parent_dir = os.path.dirname(current_dir)
os.chdir(parent_dir)
print("切换后的目录：", parent_dir)

当前工作目录： c:\Users\zhaoxs3\Documents\GitHub\NEMESIS\unit_test
切换后的目录： c:\Users\zhaoxs3\Documents\GitHub\NEMESIS


In [2]:
import pandas as pd
from nemesis.products.rates import *
from nemesis.market.indices.interest_rate_index import InterestRateIndex, DataFrameFixingSource

####################################################################
#  NEMESIS ALPHA Version 0.1.0 - This build: 24 Jan 2025 at 10:42 #
####################################################################



In [3]:
mkt_file_path = './unit_test/data/bbsw3m_curve_data_20260313.xlsx'
deposit_mkt_data = pd.read_excel(mkt_file_path, sheet_name='deposit')
swap_mkt_data = pd.read_excel(mkt_file_path, sheet_name='swap')

In [4]:
value_dt = Date(13,3,2026)

cal_type = CalendarTypes.AUSTRALIA
cal = Calendar(cal_type)

index = InterestRateIndex(
    cal_type=cal_type,
    fixing_lag=0,
    spot_lag=0,
    tenor="3M",
)

virtual_index = InterestRateIndex(
    cal_type=cal_type,
    fixing_lag=1,
    spot_lag=1,
    tenor="3M",
)

float_rate_spec = FloatRateSpec(
    multiplier=1.0,
    spread=0.0,
    compounding_type=None,
    reset_freq_type=None,
    reset_dg_type=None
)

settle_dt = cal.add_business_days(value_dt, 1)

notional = 1000000

In [5]:
deposits = [
    InterestRateDeposit(
        effective_dt=settle_dt,
        maturity_dt_or_tenor=deposit_mkt_data["Tenor"].iloc[0],
        deposit_rate=deposit_mkt_data["Rate"].iloc[0],
        dc_type=DayCountTypes.ACT_365F,
        cal_type=cal_type,
        bd_type=BusDayAdjustTypes.MODIFIED_FOLLOWING,
    )
]


In [ ]:
swaps = [
    InterestRateSwap(
        effective_dt=settle_dt,
        term_dt_or_tenor=tenor,
        fixed_leg_type=SwapTypes.PAY,
        fixed_cpn=rate,
        fixed_freq_type=FrequencyTypes.QUARTERLY,
        fixed_dc_type=DayCountTypes.ACT_365F,
        float_freq_type=FrequencyTypes.QUARTERLY,
        float_dc_type=DayCountTypes.ACT_365F,
        rate_index=virtual_index,
        rate_spec=float_rate_spec,
        notional=notional,
        payment_lag=0,
        cal_type=cal_type,
        bd_type=BusDayAdjustTypes.MODIFIED_FOLLOWING,
        dg_type=DateGenRuleTypes.BACKWARD,
        end_of_month=True
    )
    for tenor, rate in zip(swap_mkt_data["Tenor"][:6], swap_mkt_data["Rate"][:6])
] + [
    InterestRateSwap(
        effective_dt=settle_dt,
        term_dt_or_tenor=tenor,
        fixed_leg_type=SwapTypes.PAY,
        fixed_cpn=rate,
        fixed_freq_type=FrequencyTypes.SEMI_ANNUAL,
        fixed_dc_type=DayCountTypes.ACT_365F,
        float_freq_type=FrequencyTypes.QUARTERLY,
        float_dc_type=DayCountTypes.ACT_365F,
        rate_index=virtual_index,
        rate_spec=float_rate_spec,
        notional=notional,
        payment_lag=0,
        cal_type=cal_type,
        bd_type=BusDayAdjustTypes.MODIFIED_FOLLOWING,
        dg_type=DateGenRuleTypes.BACKWARD,
        end_of_month=True
    )
    for tenor, rate in zip(swap_mkt_data["Tenor"][6:], swap_mkt_data["Rate"][6:])
]


In [14]:
curve = InterestRateCurve(
    value_dt,
    deposits,
    [],
    swaps,
    interp_type=InterpTypes.LINEAR_ZERO_RATES,
    is_index=True,
    currency="AUD",
 )

In [15]:
curve.print_table(curve.pillar_dts[1:])

,Date,ZR,DF
0,2026-06-16,4.16061,0.989229
1,2026-09-16,4.26953,0.978363
2,2026-12-16,4.36938,0.967269
3,2027-03-16,4.44631,0.956161
4,2027-09-16,4.49582,0.934268
5,2028-03-16,4.52502,0.913021
6,2029-03-16,4.55857,0.871746
7,2030-03-18,4.57526,0.832133
8,2031-03-17,4.62733,0.792946
9,2032-03-16,4.67826,0.754774
